In [ ]:
"""
Fine-tune RoBERTa-large on n=100 baseline training set with CLASS WEIGHTS.
Same architecture, hyperparameters, and seed as train_roberta_final_baseline.ipynb,
but the cross-entropy loss is reweighted to address the ~9:1 class imbalance.
Saves model checkpoint for prediction.
"""

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
import os

## Config matches your CV setup
MODEL_NAME = "roberta-large"
LR = 3e-5
EPOCHS = 4
BATCH_SIZE = 32
MAX_LEN = 75
SEED = 42
OUTPUT_DIR = "model_100"

## Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
## Load training data
df = pd.read_excel("../1. train_data/train_100_baseline.xlsx")
print(f"Training on {len(df)} examples")
n_pos = int((df['nostalgic']==1).sum())
n_neg = int((df['nostalgic']==0).sum())
print(f"  Positive: {n_pos}")
print(f"  Negative: {n_neg}")
print(f"  Imbalance ratio: {n_neg/n_pos:.2f}:1")


In [ ]:
## Tokenizer
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = TextDataset(
    df["text"].tolist(),
    df["nostalgic"].astype(int).tolist(),
    tokenizer,
    MAX_LEN,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)


In [ ]:
## Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

## Class weights: inverse class frequency.
## weight on negative class = 1.0
## weight on positive class = n_neg / n_pos  (~8.9 for 50:445)
class_weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights: [neg=1.0, pos={n_neg/n_pos:.2f}]")

## Optimizer / scheduler
total_steps = len(train_loader) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

## Train
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        ## Don't pass labels to the model; compute weighted loss manually
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss = {avg:.4f}")


In [ ]:
## Save
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")


In [ ]:
## Quick sanity check: score a few sentences to confirm the model learned something
model.eval()
test_sentences = [
    "Our historic market towns and unspoiled countryside are the envy of the world.",
    "We must preserve our cultural heritage for future generations.",
    "We need to take pride in our country again and claim back our heritage.",
    "Tax policy should be revised next quarter.",
    "The new packaging act will regulate the deposit refund system.",
]
for s in test_sentences:
    enc = tokenizer(s, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding="max_length").to(device)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, dim=-1)[0,1].item()
    print(f"  P(nostalgic)={p:.3f}  '{s[:60]}'")
